(rd.phasespace)=
# 8. Particle Phase Space Coordinates

In [1]:
import numpy as np
import scibmad as sb
sb.options.display_width = 100
sb.options.display_height = 30

## Phase Space

```{figure} figures/local-coords.svg
:width: 48%

Particle coordinates are relative to the reference trajectory.
```

```{figure} figures/coordinatesystem1.svg
:width: 48%

Particle phase space.
```

The first figure shows the reference trajectory (the branch coordinate system), which was discussed
in [Machine Coordinates](coordinates.md). The branch coordinates $(x, y, z)$ associated with a given
point a distance $s$ along the reference trajectory have their origin at that point and the $z$ axis
tangent to the reference trajectory.

A particle being simulated has its own trajectory, as shown in the figure. Given a particle at some
point on its trajectory, there is a point at position $s$ on the reference trajectory such that the
$z$ coordinate of the particle is zero in the $(x, y, z)$ coordinate frame associated with that
point. With this, the particle's position and momentum $\mathbf{P}$ can be described using the
coordinates

$$
(x(s), y(s), P_x(s), P_y(s), P_z(s), t(s))
$$

where $t(s)$ is the time at which the particle is at the given point. From now on, to simplify the
notation, the $s$ dependence will be dropped.

For tracking, canonical phase space coordinates are used, with the convention that an upper case
$P$ denotes the (unnormalized) momentum and a lower case $p$ the phase space momentum. The phase
space coordinates are

$$
(x, p_x, y, p_y, z, p_z)
$$

where

$$
p_x = \frac{P_x}{P_0}, \quad p_y = \frac{P_y}{P_0}, \quad p_z = \frac{P - P_0}{P_0}, \quad
z = -\beta c\,(t - t_\mathrm{ref})
$$

with

- $P_0$ the reference momentum, see [Reference Energy](#rd.refenergy).
- $\beta c$ the velocity of the particle.
- $t_\mathrm{ref}(s)$ the time at which the reference particle reaches the point $s$. The reference
  particle is a fictitious particle that can be imagined to travel on the reference trajectory.
  Frequently, this reference particle is thought of as describing the center of a bunch of particles.

Notes:

- Do not confuse the canonical $z$ coordinate with the $z$ coordinate of the particle in the
  $(x, y, z)$ coordinate frame. By construction, the latter is always zero.
- For a bunch of particles at a given $s$ position, in general, the particles will have differing
  times $t$.
- If the reference particle has the same $\beta$ as a particle, the canonical $z$ is the
  longitudinal distance of the particle from the reference particle. Positive $z$ means that the
  particle is in front of the reference particle.
- $p_z$ is also called $\delta$.

## Example

A beamline with a bend and a quadrupole, for positrons with 10 MeV total energy:

In [2]:
import numpy as np
import scibmad as sb

with sb.elements():
    b = sb.SBend(L=0.5, angle=0.5)  # g_ref = 1/bending radius = 1 1/m
    q = sb.Quadrupole(L=0.6, Kn1=10.0)
lat = sb.Beamline([b, q], species_ref=sb.Species("positron"), E_ref=10e6)

A particle is tracked through it with `track`, starting with $y$ = 1 cm, $p_x$ = 0.06 and
$p_z$ = −0.2, with spin tracking turned on (`spin=True`). The coordinates are saved at every
integration step ({jl}`at=:`{py}`at=...`), and, with more integration steps, also inside the
elements:

In [3]:
b.n_steps = 20
q.n_steps = 20
v0 = [[0, 0.06, 0.01, 0, 0, -0.2]]  # x, px, y, py, z, pz
res = sb.track(lat, v0=v0, spin=True, at=...)
res

TrackResult of 1 particle:
config:
 n_turns                         = 1
 save_every_n_turns              = 1
 in_body_coordinates             = false
 save_reference                  = false
 scalar_params                   = false
 rf_on                           = true
 ramp_particle_energy_without_rf = false
 ramp_update_each_particle       = false
 batch_start                     = 1
 verbose                         = false
 groupsize                       = 0
 use_cpu_multithreading          = false
 use_KA                          = false
 use_explicit_SIMD               = true

elapsed_time = 0.000116072 seconds

start = 1
turns = [0, 1]

s, index, name, kind    the saved locations: 41 locations, from b (s = 0.0) to the end of the beamline (s = 1.1)
state                   particle state codes, e.g. state(location, turn=n, particle=i)
x, px, y, py, z, pz     the coordinates, e.g. x(location, turn=n, particle=i)
v                       all six coordinates, e.g. v(location, turn=n

The result holds the particle at the saved locations: for example its coordinates at the end of
the beamline after the first (and only) turn:

In [4]:
res.v(-1, turn=1, particle=1)

array([-0.01591224,  0.01162814,  0.03867606,  0.10490576, -0.00688104,
       -0.2       ])

Spin tracking tracks the rotation of the spin as a quaternion, so that the spin can be computed
afterwards for any initial spin. With the initial spin along $x$:

In [5]:
s = sb.track_spin(res.q, [1, 0, 0])
s(-1, turn=1, particle=1)  # The spin (x, y, z) at the end

array([0.99752506, 0.00573158, 0.07007788])

The orbit, with $p_x$ = 0.06 at the start and changed to $p_x$ = 0.1:

In [6]:
from matplotlib.figure import Figure
fig = Figure(figsize=(7, 3.5), layout="constrained")
ax = fig.subplots()
ax.plot(res.s, 1e3*res.x(..., turn=1, particle=1), label="x")
ax.plot(res.s, 1e3*res.y(..., turn=1, particle=1), label="y")
res2 = sb.track(lat, v0=[[0, 0.1, 0.01, 0, 0, -0.2]], spin=True, at=...)
ax.plot(res2.s, 1e3*res2.x(..., turn=1, particle=1), label="x (px = 0.1)", linestyle="--")
ax.set_xlabel("s [m]")
ax.set_ylabel("[mm]")
ax.legend(loc="upper left")
fig

<Figure size 700x350 with 1 Axes>

`res.s` is the $s$ position of every saved location, and `res.x[:, turn=1, particle=1]` is $x$ at
all of them. See [Track](../track.md) for everything a `TrackResult` holds.

## Exercises

1. **Phase space:** Construct a beamline with a single drift of 2 m length. Set the reference
   particle to be a proton with a reference energy of $10^{12}$ eV (so that the approximation that
   the particle velocity is the speed of light can be made), and set the initial particle position
   to $p_x$ = 0.2, $p_z$ = 1 with all other coordinates zero. Calculate from first principles what
   the phase space coordinates will be at the end of the beamline, and check that your answer
   agrees with `track`.

2. **Fringe fields:** Quadrupoles have hard-edge fringe fields at both ends by default
   (`q.fringe_at` is `Fringe.BothEnds`). Track the particle of the example again with
   `q.fringe_at = Fringe.NoEnd`. How much do the fringe fields change the orbit at the end? Why are
   they so important here?